<link rel='stylesheet' type='text/css' href='./style.css'>

<div class='tutorial-header'>

## Tutorial \#2: Execution

</div>

<link rel='stylesheet' type='text/css' href='./style.css'>

<div class='tutorial-text'>

<b><i>Spark</i></b>, is built on top of Jax and the one of the core features of Jax is Just-In-Time compilation, or JIT for short. JIT is extremely good at optimizing how your code executes. Therefore, it is important to have a minimum understanding of how to play with JIT. So, let's start with a simple JIT example.

</div>

In [1]:
import jax
import jax.numpy as jnp

# Some really cool function
def my_awesome_function(x, y):
    return x * y + x**2

# Two random arrays
rng = jax.random.key(42)
key_x, key_y = jax.random.split(rng, 2)
x = jax.random.uniform(key_x, shape=(4096,4096), dtype=jnp.float32)
y = jax.random.uniform(key_y, shape=(4096,4096), dtype=jnp.float32)

# Test it!
%timeit my_awesome_function(x, y)

1.41 ms ± 1.09 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


<link rel='stylesheet' type='text/css' href='./style.css'>

<div class='tutorial-text'>

Now, this is fast, but can it be better?

Let's JIT the funtion and run it again.

</div>

In [2]:
# Some really cool and fast function
@jax.jit
def my_awesome_function(x, y):
    return x * y + x**2

# We perform a dummy call first in order to build the jitted function.
# NOTE: This is usually not necessary unless you want to benchmark your code
my_awesome_function(x, y)

# Now it should be faster
%timeit my_awesome_function(x, y).block_until_ready()

700 μs ± 49.1 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


<link rel='stylesheet' type='text/css' href='./style.css'>

<div class='tutorial-text'>

Note that, JIT as its name suggest, compiles the code just before running it, this leads to longer execution times the first the function is executed. Depending on the code being JITted this can lead to really long compilation times, for example, it is possible to compile and entire for loop that executes thousands of operations in total (note that this is not recommended but may be a useful trick depending on the circumstances). 

Another important gotcha moment is when we try to pass some constant parameters to a function that we want to jit.

</div>

In [3]:
@jax.jit
def repetitive_add(x, n):
    print(f'--- TRACING: n = {n} ---')		# <--- Print statements on jitted functions, only are shown the first time they are called. 
    for _ in range(n):
        x = x + 1.0
    return x

try:
	print(f'Result: {repetitive_add(10.0, 5)}')
except:
	print('Oh no!, Something went wrong!')

--- TRACING: n = JitTracer<~int32[]> ---
Oh no!, Something went wrong!


<link rel='stylesheet' type='text/css' href='./style.css'>

<div class='tutorial-text'>

Unless other way specified, JIT is going to treat every single argument as a valid array. This is problematic when, for example, we wish to compile a function with a loop, as the previous example, or some other variable type that we do not wish to use as tensor (or more specifically a pytree) inside the function. These problem are usually solved by composing our decorator with <b>functools.partial</b> and indicating which arguments are static, either by position or name.

</div>

In [4]:
from functools import partial

@partial(jax.jit, static_argnums=(1,))
def repetitive_add_argnums(x, n):
    print(f'--- TRACING: n = {n} ---')			
    for _ in range(n):
        x = x + 1.0
    return x

# Execute function
print(f'Result argnums: {repetitive_add_argnums(10.0, 5)}')


# Alternatively,
@partial(jax.jit, static_argnames=('n',))
def repetitive_add_argnames(x, n):
    print(f'--- TRACING: n = {n} ---')			
    for _ in range(n):
        x = x + 1.0
    return x

# Execute function
print(f'Result argnames: {repetitive_add_argnames(10.0, 5)}')

--- TRACING: n = 5 ---
Result argnums: 15.0
--- TRACING: n = 5 ---
Result argnames: 15.0


<link rel='stylesheet' type='text/css' href='./style.css'>

<div class='tutorial-text'>

The other common gotcha moment is when we accidentally trigger the compiler over and over again. Sometimes, when something like this happens, it may be possible to refactor the code such that the jitted function does not depend on the specific parameter that is triggering the JIT compiler. However, most of the time this would depend on a case-to-case basis in which compromises are unavoidable.

</div>

In [5]:
# Triggers jit
print(f'First call (n = 8): {repetitive_add_argnums(10.0, 8)}')
# Uses cache
print(f'Second call (n = 8): {repetitive_add_argnums(10.0, 8)}')
# Triggers jit again!
print(f'Third call (n = 7): {repetitive_add_argnums(10.0, 7)}\n')


# The not so fast but necessary compromise
@jax.jit
def single_add(x):
	print(f'--- TRACING ---')			
	return x + 1.0

def compromise_add(x, n):
	for _ in range(n):
		x = single_add(x)
	return x


# Triggers in the inner methodjit
print(f'First call (n = 3): {compromise_add(10.0, 3)}')

print(f'Second call (n = 4): {compromise_add(10.0, 4)}')

print(f'Third call (n = 5): {compromise_add(10.0, 4)}')

--- TRACING: n = 8 ---
First call (n = 8): 18.0
Second call (n = 8): 18.0
--- TRACING: n = 7 ---
Third call (n = 7): 17.0

--- TRACING ---
First call (n = 3): 13.0
Second call (n = 4): 14.0
Third call (n = 5): 14.0


<link rel='stylesheet' type='text/css' href='./style.css'>

<div class='tutorial-text'>

Unfortunately, it is slightly more complicated in <b><i>Spark</i></b> (but do not worry, it is not that complicated!).

Under the hood Jax unfolds the computation graph and tries to optimize it, this however is not trivial when you have to do some state management. In <b><i>Spark</i></b> we borrow one of the core element of Flax, an excellent machine learning library, that allow us to not care to much about state management at the expense of a slightly more complicated JIT execution. Although this may change as <b><i>Spark</i></b> matures, right now this is the standard approach to execute your code.

Let's start by initializing some simple ALIF model.

</div>

In [6]:
# Imports
import spark

# Number of neurons
units = (8,)

# Input shape
input_shape = (16,)

# Model initialization
alif_neurons = spark.nn.neurons.ALIFNeuron(
    units=units,
	inhibitory_rate=0.2,
)

# Test the model.
rng = jax.random.key(42)
rng, key = jax.random.split(rng, 2)
in_spikes = spark.SpikeArray( 
    jax.random.uniform(key, shape=input_shape, dtype=jnp.float16) < 0.5 
)

# Note that the model is not completely built before the first time is called.
try:
    alif_neurons.synapses.kernel
    print('This is a really nice kernel!.\n')
except:
    print('Oh no!, alif_neurons do not have a kernel property.\n')

# First call to alif_neurons 
out_spikes = alif_neurons(in_spikes=in_spikes)
print(out_spikes, '\n')

# Now the kernel property exists.
try:
    alif_neurons.synapses.kernel
    print('This is a really nice kernel!.\n')
except:
    print('Oh no!, alif_neurons do not have a kernel property.\n')

Oh no!, alif_neurons do not have a kernel property.

{'out_spikes': SpikeArray(_encoding=Array([0, 0, 0, 0, 2, 0, 0, 0], dtype=uint8), async_spikes=False)} 

This is a really nice kernel!.



<link rel='stylesheet' type='text/css' href='./style.css'>

<div class='tutorial-text'>

Similarly, to the cases above, we just need to wrap the execution of the model inside a function that we can then compile. Remember that the first time the function is executed it can take a little bit more time (important if you want to benchmark your own models!).

</div>

In [ ]:
# JITted function for execution
@spark.jit						# <-- Convinience wrapper of nnx.jit
def run_model(model, **inputs):
	outputs = model(**inputs)
	return outputs, model		# <-- Return and replace the model. Otherwise the model will not change

# Execution
outputs, alif_neurons = run_model(alif_neurons, in_spikes=in_spikes) # Note that we overwrite the model after the execution.

%timeit run_model(alif_neurons, in_spikes=in_spikes)

2.96 ms ± 27.9 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


<link rel='stylesheet' type='text/css' href='./style.css'>

<div class='tutorial-text'>

Although we can feed directly our model to JAX and it will execute fine, it is almost always faster to split the model into a graph and a state (quirks of the code ¯\\_(ツ)_/¯).

To separate the logic of our model into a graph and an state we can use <b>spark.split</b> and to glue it back for execution we use <b>spark.merge</b>. Note that currently this is just a convinience wrapper arounds Flax's nnx.split and nnx.merge, respectively, although it may change in the future. That's it, this is as much as JIT as you need to know. Just remember, when you define your own models use <b>spark.Constant</b>  and <b>spark.Variable</b> to wrap your code, otherwise JIT is going to complain!.

As a small observation, note that we feed everything to the model with <b>**inputs</b> since the <b>\_\_call\_\_</b> method only allows for keyworded arguments (which can still be received by position). This could be really advantageous when we encounter models that have more than one input stream or we have many different models that we want to try since it allows us to create code that is more reusable and flexible.

</div>

In [ ]:
# JITted function for execution
@jax.jit
def run_model_split(graph, state, **inputs):
	model = spark.merge(graph, state)	# Merge the graph and the state into a single executable model equivalent to ALIFNeuron.
	outputs = model(**inputs)
	_, state = spark.split((model))		# Split the model back into a graph and a state. Here we are discarding the graph since typically it doesn't change.
	return outputs, state

# Execution
graph, state = spark.split((alif_neurons))	# <-- Note that model is inside another parenthesis, this will be important for more complex cases.
outputs, state = run_model_split(graph, state, in_spikes=in_spikes)	# Note that we overwrite the state after the execution.

%timeit run_model_split(graph, state, in_spikes=in_spikes)

103 μs ± 2.07 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


<link rel='stylesheet' type='text/css' href='./style.css'>

<div class='tutorial-text'>

This is basically all the JAX & JIT you need to know to get the most out of <b><i>Spark</i></b>!

If you wish to know more about Jax and JIT, you may take a look at [JIT Compilation](https://docs.jax.dev/en/latest/jit-compilation.html) and [🔪 JAX - The Sharp Bits 🔪](https://docs.jax.dev/en/latest/notebooks/Common_Gotchas_in_JAX.html).

</div>